# Tarea 1 – Parte 2: Modelado y Clasificación

**Medio:** El Comercio  
**Número de noticias:** 360

## Objetivo

Comparar tres estrategias de clasificación automática de titulares:

1. Zero-shot.
2. Supervisión débil mediante Gemini.
3. Fine-tuning de DistilBERT.

Las categorías utilizadas son:

- politica
- economia
- deportes
- ciencia_tecnologia

Las métricas principales serán Accuracy y F1-macro.

In [14]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATOS = Path("datos")
MODELO = Path("modelo")

DATOS.mkdir(exist_ok=True)
MODELO.mkdir(exist_ok=True)

In [15]:
df = pd.read_csv(DATOS / "noticias.csv")

print("Dimensión:", df.shape)
print()

print(df["etiqueta"].value_counts())
print()

print("Valores nulos:", df.isna().sum().sum())
print("URLs duplicadas:", df["url"].duplicated().sum())

df.head()

Dimensión: (360, 7)

etiqueta
politica              95
economia              95
ciencia_tecnologia    95
deportes              75
Name: count, dtype: int64

Valores nulos: 0
URLs duplicadas: 0


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Presidente del TC sobre leyes aprobadas por el...,El presidente del Tribunal Constitucional (TC)...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/presid...,2026-09-07T20:51:41-05:00,politica
1,Caso Árbitros-Odebrecht: audiencia de control ...,La Corte Superior Nacional de Justicia Penal E...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/caso-a...,2026-10-12T12:48:23-05:00,politica
2,Flavio Figallo del Partido del Buen Gobierno: ...,"El senador del Partido del Buen Gobierno, Flav...",El Comercio,Congreso,https://elcomercio.pe/politica/congreso/flavio...,2026-10-16T11:47:20-05:00,politica
3,Unión Europea y Minem acuerdan creación de mes...,La Unión Europea (UE) y el Ministerio de Energ...,El Comercio,Gobierno,https://elcomercio.pe/politica/gobierno/union-...,2026-10-13T19:19:04-05:00,politica
4,Fiscalía realizó diligencias en Ministerio de ...,Las diligencias estuvieron a cargo de la fisca...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/albert...,2026-08-27T13:43:00-05:00,politica


## 1. Baseline Zero-Shot

Se seleccionó una muestra balanceada de 40 titulares,
con 10 noticias por categoría.

El modelo zero-shot clasifica los titulares sin haber sido
entrenado previamente con la base construida para este proyecto.

In [16]:
muestra_zero_shot = (
    df.groupby("etiqueta", group_keys=False)
      .sample(n=10, random_state=SEED)
      .reset_index(drop=True)
)

print("Tamaño:", len(muestra_zero_shot))
print()

print(muestra_zero_shot["etiqueta"].value_counts())

Tamaño: 40

etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64


In [17]:
from transformers import pipeline

clasificador_zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

etiquetas_modelo = {
    "politica": "política",
    "economia": "economía y negocios",
    "deportes": "deportes",
    "ciencia_tecnologia": "ciencia y tecnología"
}

candidate_labels = list(etiquetas_modelo.values())

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 17750.88it/s]


In [18]:
predicciones_zero_shot = []
confianzas_zero_shot = []

for i, titular in enumerate(muestra_zero_shot["titular"]):

    resultado = clasificador_zero_shot(
        titular,
        candidate_labels=candidate_labels,
        hypothesis_template="Este titular pertenece a la categoría {}."
    )

    prediccion_texto = resultado["labels"][0]
    confianza = resultado["scores"][0]

    etiqueta_predicha = next(
        clave
        for clave, valor in etiquetas_modelo.items()
        if valor == prediccion_texto
    )

    predicciones_zero_shot.append(etiqueta_predicha)
    confianzas_zero_shot.append(confianza)

    print(
        f"{i + 1}/40 -> "
        f"{etiqueta_predicha} "
        f"({confianza:.3f})"
    )

muestra_zero_shot["pred_zero_shot"] = predicciones_zero_shot
muestra_zero_shot["confianza_zero_shot"] = confianzas_zero_shot

1/40 -> ciencia_tecnologia (0.386)
2/40 -> politica (0.307)
3/40 -> politica (0.506)
4/40 -> deportes (0.318)
5/40 -> economia (0.370)
6/40 -> ciencia_tecnologia (0.626)
7/40 -> ciencia_tecnologia (0.461)
8/40 -> politica (0.374)
9/40 -> politica (0.993)
10/40 -> economia (0.729)
11/40 -> deportes (0.982)
12/40 -> politica (0.635)
13/40 -> deportes (0.994)
14/40 -> deportes (0.997)
15/40 -> deportes (0.955)
16/40 -> deportes (0.528)
17/40 -> deportes (0.981)
18/40 -> deportes (0.497)
19/40 -> deportes (0.974)
20/40 -> deportes (0.998)
21/40 -> economia (0.856)
22/40 -> politica (0.587)
23/40 -> economia (0.991)
24/40 -> ciencia_tecnologia (0.627)
25/40 -> economia (0.978)
26/40 -> politica (0.755)
27/40 -> economia (0.827)
28/40 -> politica (0.936)
29/40 -> politica (0.679)
30/40 -> economia (0.974)
31/40 -> politica (0.913)
32/40 -> politica (0.940)
33/40 -> politica (0.991)
34/40 -> politica (0.503)
35/40 -> politica (0.998)
36/40 -> politica (0.960)
37/40 -> politica (0.908)
38/40 -

In [19]:
accuracy_zero_shot = accuracy_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["pred_zero_shot"]
)

f1_zero_shot = f1_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["pred_zero_shot"],
    average="macro"
)

print("Accuracy Zero-Shot:", round(accuracy_zero_shot, 4))
print("F1-macro Zero-Shot:", round(f1_zero_shot, 4))

Accuracy Zero-Shot: 0.65
F1-macro Zero-Shot: 0.6328


## 2. Supervisión débil con Ollama

Se utiliza el modelo local Qwen 2.5 3B mediante Ollama para clasificar los 360 titulares
en una de las cuatro categorías permitidas:

- politica
- economia
- deportes
- ciencia_tecnologia

Para cada titular se registra la etiqueta predicha y una confianza estimada.

In [24]:
import ollama
import json
import time

In [25]:
import unicodedata

ETIQUETAS_VALIDAS = {
    "politica",
    "economia",
    "deportes",
    "ciencia_tecnologia"
}

def limpiar_etiqueta(texto):

    if texto is None:
        return None

    texto = str(texto).strip().lower()

    # Quitar tildes
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    texto = texto.replace("-", "_")
    texto = texto.replace(" ", "_")

    equivalencias = {
        "politica": "politica",
        "economia": "economia",
        "deporte": "deportes",
        "deportes": "deportes",
        "tecnologia": "ciencia_tecnologia",
        "ciencia": "ciencia_tecnologia",
        "ciencia_tecnologia": "ciencia_tecnologia",
        "ciencia_y_tecnologia": "ciencia_tecnologia"
    }

    return equivalencias.get(texto)

In [28]:
ruta_llm = DATOS / "etiquetas_llm.csv"

resultados_llm = []

for i, fila in df.iterrows():

    try:
        etiqueta_llm, confianza = clasificar_con_ollama(
            fila["titular"]
        )

        resultados_llm.append({
            "titular": fila["titular"],
            "etiqueta_llm": etiqueta_llm,
            "confianza": confianza,
            "modelo_llm": "qwen2.5:3b",
            "fecha": fila["fecha"]
        })

    except Exception as e:

        print(
            f"Error en noticia {i + 1}:",
            str(e)[:120]
        )

        resultados_llm.append({
            "titular": fila["titular"],
            "etiqueta_llm": None,
            "confianza": None,
            "modelo_llm": "qwen2.5:3b",
            "fecha": fila["fecha"]
        })

    # Guardar avance cada 20 noticias
    if (i + 1) % 20 == 0:

        pd.DataFrame(
            resultados_llm
        ).to_csv(
            ruta_llm,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Guardadas {i + 1} de {len(df)} noticias"
        )

# Guardado final
pd.DataFrame(
    resultados_llm
).to_csv(
    ruta_llm,
    index=False,
    encoding="utf-8-sig"
)

print("Etiquetado terminado.")

Error en noticia 1: name 'clasificar_con_ollama' is not defined
Error en noticia 2: name 'clasificar_con_ollama' is not defined
Error en noticia 3: name 'clasificar_con_ollama' is not defined
Error en noticia 4: name 'clasificar_con_ollama' is not defined
Error en noticia 5: name 'clasificar_con_ollama' is not defined
Error en noticia 6: name 'clasificar_con_ollama' is not defined
Error en noticia 7: name 'clasificar_con_ollama' is not defined
Error en noticia 8: name 'clasificar_con_ollama' is not defined
Error en noticia 9: name 'clasificar_con_ollama' is not defined
Error en noticia 10: name 'clasificar_con_ollama' is not defined
Error en noticia 11: name 'clasificar_con_ollama' is not defined
Error en noticia 12: name 'clasificar_con_ollama' is not defined
Error en noticia 13: name 'clasificar_con_ollama' is not defined
Error en noticia 14: name 'clasificar_con_ollama' is not defined
Error en noticia 15: name 'clasificar_con_ollama' is not defined
Error en noticia 16: name 'clasifi

In [29]:
df_llm = pd.read_csv(
    DATOS / "etiquetas_llm.csv"
)

print("Dimensión:", df_llm.shape)

print("\nDistribución:")
print(
    df_llm["etiqueta_llm"]
    .value_counts(dropna=False)
)

print(
    "\nEtiquetas vacías:",
    df_llm["etiqueta_llm"]
    .isna()
    .sum()
)

print(
    "Confianzas vacías:",
    df_llm["confianza"]
    .isna()
    .sum()
)

Dimensión: (360, 5)

Distribución:
etiqueta_llm
NaN    360
Name: count, dtype: int64

Etiquetas vacías: 360
Confianzas vacías: 360


In [30]:
import unicodedata

def normalizar_etiqueta(texto):

    if texto is None:
        return None

    texto = str(texto).strip().lower()

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    # Aceptar pequeñas variaciones del modelo
    if "politic" in texto:
        return "politica"

    if "econom" in texto or "negocio" in texto:
        return "economia"

    if "deport" in texto:
        return "deportes"

    if "tecnolog" in texto or "ciencia" in texto:
        return "ciencia_tecnologia"

    return None


def clasificar_con_ollama_estricto(titular):

    prompt = f"""
Clasifica este titular en EXACTAMENTE una de estas cuatro etiquetas:

politica
economia
deportes
ciencia_tecnologia

No inventes otras categorías.

Devuelve solamente JSON:

{{
  "etiqueta": "politica",
  "confianza": 0.95
}}

Titular:
{titular}
"""

    respuesta = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        format="json",
        options={"temperature": 0}
    )

    resultado = json.loads(
        respuesta["message"]["content"]
    )

    etiqueta = normalizar_etiqueta(
        resultado.get("etiqueta")
    )

    confianza = float(
        resultado.get("confianza", 0)
    )

    return etiqueta, confianza

In [31]:
print("Dimensión:", df_llm.shape)

print("\nDistribución:")
print(
    df_llm["etiqueta_llm"]
    .value_counts(dropna=False)
)

print(
    "\nEtiquetas vacías:",
    df_llm["etiqueta_llm"].isna().sum()
)

print(
    "Confianzas vacías:",
    df_llm["confianza"].isna().sum()
)

Dimensión: (360, 5)

Distribución:
etiqueta_llm
NaN    360
Name: count, dtype: int64

Etiquetas vacías: 360
Confianzas vacías: 360


In [33]:
import json
import unicodedata
import ollama

def normalizar_etiqueta(texto):
    if texto is None:
        return None

    texto = str(texto).strip().lower()

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    if "politic" in texto:
        return "politica"

    if "econom" in texto or "negocio" in texto:
        return "economia"

    if "deport" in texto:
        return "deportes"

    if "tecnolog" in texto or "ciencia" in texto:
        return "ciencia_tecnologia"

    return None


def corregir_con_ollama(titular):

    prompt = f"""
Debes clasificar este titular.

Solo puedes responder con UNA de estas cuatro etiquetas exactas:

politica
economia
deportes
ciencia_tecnologia

Devuelve solamente JSON con este formato:

{{
    "etiqueta": "politica",
    "confianza": 0.95
}}

Titular:
{titular}
"""

    respuesta = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        format="json",
        options={"temperature": 0}
    )

    resultado = json.loads(
        respuesta["message"]["content"]
    )

    etiqueta = normalizar_etiqueta(
        resultado.get("etiqueta")
    )

    confianza = float(
        resultado.get("confianza", 0)
    )

    return etiqueta, confianza


# SOLO las filas que quedaron vacías
indices = df_llm[
    df_llm["etiqueta_llm"].isna()
].index.tolist()

print("Noticias pendientes:", len(indices))

for numero, idx in enumerate(indices, start=1):

    titular = df_llm.loc[idx, "titular"]

    etiqueta = None
    confianza = None

    # Hasta 3 intentos para esa noticia
    for intento in range(3):

        try:
            etiqueta, confianza = corregir_con_ollama(titular)

            if etiqueta is not None:
                break

        except Exception:
            pass

    if etiqueta is not None:
        df_llm.loc[idx, "etiqueta_llm"] = etiqueta
        df_llm.loc[idx, "confianza"] = confianza

    print(
        f"{numero}/{len(indices)} -> {etiqueta}"
    )


# Guardar el archivo corregido
df_llm.to_csv(
    DATOS / "etiquetas_llm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCorrección terminada.")

Noticias pendientes: 360


TypeError: Invalid value 'politica' for dtype 'float64'

In [ ]:
print("Etiquetas vacías:", df_llm["etiqueta_llm"].isna().sum())

print("\nDistribución:")
print(
    df_llm["etiqueta_llm"].value_counts(dropna=False)
)

Etiquetas vacías: 1

Distribución:
etiqueta_llm
politica              144
economia               78
ciencia_tecnologia     75
deportes               62
NaN                     1
Name: count, dtype: int64


In [ ]:
pendiente = df_llm[
    df_llm["etiqueta_llm"].isna()
]

pendiente[["titular", "confianza"]]

NameError: name 'df_llm' is not defined

In [ ]:

idx = pendiente.index[0]

titular_pendiente = df_llm.loc[idx, "titular"]

print(titular_pendiente)

Informe Planeta Vivo 2026: la fauna silvestre cae 73% y América Latina es la región más golpeada (95 %)


In [ ]:
idx = pendiente.index[0]

df_llm.loc[idx, "etiqueta_llm"] = "ciencia_tecnologia"

In [ ]:
df_llm.to_csv(
    DATOS / "etiquetas_llm.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(
    "Etiquetas vacías:",
    df_llm["etiqueta_llm"].isna().sum()
)

NameError: name 'df_llm' is not defined

In [ ]:
auditoria = (
    df_llm
    .sample(n=20, random_state=42)
    .reset_index(drop=True)
)

auditoria["etiqueta_humana"] = ""

auditoria[
    [
        "titular",
        "etiqueta_llm",
        "confianza",
        "etiqueta_humana"
    ]
]

NameError: name 'df_llm' is not defined

In [ ]:
auditoria_revision = auditoria.merge(
    df[["titular", "bajada"]],
    on="titular",
    how="left"
)

pd.set_option("display.max_colwidth", None)

auditoria_revision[
    [
        "titular",
        "bajada",
        "etiqueta_llm",
        "confianza"
    ]
]

,titular,bajada,etiqueta_llm,confianza
0,El Niño en Perú: no está operativo ninguno de los 18 sistemas de alertas tempranas previstos desde 2017,"Ese año, el Estado prometió que siete regiones podrían detectar lluvias intensas y advertir a la población ante posibles inundaciones, deslizamientos o activación de quebradas. Casi 10 años después, de los 18 Sistemas de Alerta Temprana previstos, ninguno está operativo y dos fueron descartados por falta de presupuesto.",politica,0.95
1,Presidente del TC: “No puedes formalizar a costa del medio ambiente” | ENTREVISTA,"Pedro Hernández Chávez, nuevo presidente del Tribunal Constitucional, analiza las decisiones del TC, el caso Vladimir Cerrón, el futuro del Reinfo, las facultades del Congreso y los retos que enfrenta el máximo intérprete de la Constitución.",politica,0.95
2,Alianza Lima vs. Atlético Grau: minuto a minuto del partido por Liga 1,Sigue la transmisión de Alianza Lima vs. Atlético Grau en vivo online por la fecha 12 del Torneo Clausura de la Liga 1.,deportes,0.95
3,"De Kluiverth en Alianza a Osorio de la ‘U’: la Liga 1 vendió 18 promesas a Europa desde 2020, pero casi la mitad ya regresó al fútbol peruano","Desde la venta de Kluiverth Aguilar en 2020 hasta el reciente fichaje de Sebastián Osorio por Sporting de Portugal, 18 futbolistas dejaron la Liga 1 directamente rumbo a Europa. Ocho regresaron al torneo peruano.",deportes,0.95
4,Fujimori,"La presidenta de la República, Keiko Fujimori, advirtió que el Perú afrontará épocas difíciles por los efectos del fenómeno El Niño. La mandataria señaló que el calentamiento de las aguas marinas impacta de forma directa en las actividades productivas de la costa nacional.",politica,0.95
5,"Quién es Francis Halzen, el Nobel que dedica su vida al observatorio de neutrinos a escala kilométrica en Polo Sur","Nacido en 1944 en Tienen, en Bélgica, Halzen es el investigador principal de IceCube, profesor de investigación Vilas y profesor Gregory Breit en la Universidad de Wisconsin-Madison, en EE.UU.",ciencia_tecnologia,0.95
6,Eliane Karp reitera pedido a Keiko Fujimori de gracia presidencial para Alejandro Toledo,"Eliane Karp, esposa del expresidente Alejandro Toledo, reiteró su pedido a la presidenta de la República, Keiko Fujimori, para que otorgue una gracia presidencial a su esposo, quien permanece recluido en el penal de Barbadillo.",politica,0.95
7,Guillermo Bermejo: Defensoría del Pueblo pide informes al PJ alegando demoras en recurso de sentenciado por afiliación al terrorismo,"A través de un oficio, la defensoría solicitó que la Corte Suprema le informe las razones por las que el recurso del excongresista, sentenciado por afiliación al terrorismo, no ha tenido respuesta alguna.",politica,0.95
8,Meta presenta nuevo modelo de IA y afirma que está al nivel de OpenAI y Anthropic,Muse Wonder ha sido calificado entre los modelos más eficientes del mercado. Está disponible para desarrolladores y hará funcionar “pronto” al asistente Muse,ciencia_tecnologia,0.95
9,AFP y ONP: Los cuatro cambios que aplica la nueva ley de reforma,"La normativa finalmente fue promulgada por el Ejecutivo hoy y entra en vigor una vez se publique su reglamento. Para ello, el Gobierno tiene un plazo de 180 días hábiles.",politica,0.95


In [ ]:
etiquetas_humanas = [
    "politica",              # 0
    "politica",              # 1
    "deportes",              # 2
    "deportes",              # 3
    "politica",              # 4
    "ciencia_tecnologia",    # 5
    "politica",              # 6
    "politica",              # 7
    "ciencia_tecnologia",    # 8
    "economia",              # 9
    "ciencia_tecnologia",    # 10
    "deportes",              # 11
    "ciencia_tecnologia",    # 12
    "deportes",              # 13
    "economia",              # 14
    "economia",              # 15
    "ciencia_tecnologia",    # 16
    "deportes",              # 17
    "economia",              # 18
    "deportes"               # 19
]

auditoria_revision["etiqueta_humana"] = etiquetas_humanas

auditoria_revision[
    [
        "titular",
        "etiqueta_llm",
        "etiqueta_humana"
    ]
]

,titular,etiqueta_llm,etiqueta_humana
0,El Niño en Perú: no está operativo ninguno de los 18 sistemas de alertas tempranas previstos desde 2017,politica,politica
1,Presidente del TC: “No puedes formalizar a costa del medio ambiente” | ENTREVISTA,politica,politica
2,Alianza Lima vs. Atlético Grau: minuto a minuto del partido por Liga 1,deportes,deportes
3,"De Kluiverth en Alianza a Osorio de la ‘U’: la Liga 1 vendió 18 promesas a Europa desde 2020, pero casi la mitad ya regresó al fútbol peruano",deportes,deportes
4,Fujimori,politica,politica
5,"Quién es Francis Halzen, el Nobel que dedica su vida al observatorio de neutrinos a escala kilométrica en Polo Sur",ciencia_tecnologia,ciencia_tecnologia
6,Eliane Karp reitera pedido a Keiko Fujimori de gracia presidencial para Alejandro Toledo,politica,politica
7,Guillermo Bermejo: Defensoría del Pueblo pide informes al PJ alegando demoras en recurso de sentenciado por afiliación al terrorismo,politica,politica
8,Meta presenta nuevo modelo de IA y afirma que está al nivel de OpenAI y Anthropic,ciencia_tecnologia,ciencia_tecnologia
9,AFP y ONP: Los cuatro cambios que aplica la nueva ley de reforma,politica,economia


In [ ]:
print(auditoria_revision.columns.tolist())

['titular', 'etiqueta_llm', 'confianza', 'modelo_llm', 'fecha', 'etiqueta_humana', 'bajada']


In [ ]:
auditoria_revision[
    [
        "titular",
        "etiqueta_llm",
        "etiqueta_humana"
    ]
]

,titular,etiqueta_llm,etiqueta_humana
0,El Niño en Perú: no está operativo ninguno de los 18 sistemas de alertas tempranas previstos desde 2017,politica,politica
1,Presidente del TC: “No puedes formalizar a costa del medio ambiente” | ENTREVISTA,politica,politica
2,Alianza Lima vs. Atlético Grau: minuto a minuto del partido por Liga 1,deportes,deportes
3,"De Kluiverth en Alianza a Osorio de la ‘U’: la Liga 1 vendió 18 promesas a Europa desde 2020, pero casi la mitad ya regresó al fútbol peruano",deportes,deportes
4,Fujimori,politica,politica
5,"Quién es Francis Halzen, el Nobel que dedica su vida al observatorio de neutrinos a escala kilométrica en Polo Sur",ciencia_tecnologia,ciencia_tecnologia
6,Eliane Karp reitera pedido a Keiko Fujimori de gracia presidencial para Alejandro Toledo,politica,politica
7,Guillermo Bermejo: Defensoría del Pueblo pide informes al PJ alegando demoras en recurso de sentenciado por afiliación al terrorismo,politica,politica
8,Meta presenta nuevo modelo de IA y afirma que está al nivel de OpenAI y Anthropic,ciencia_tecnologia,ciencia_tecnologia
9,AFP y ONP: Los cuatro cambios que aplica la nueva ley de reforma,politica,economia


In [ ]:
auditoria_revision["acuerdo"] = (
    auditoria_revision["etiqueta_llm"]
    == auditoria_revision["etiqueta_humana"]
)

acuerdo_llm = auditoria_revision["acuerdo"].mean()

print(
    "Acuerdo humano-LLM:",
    round(acuerdo_llm * 100, 2),
    "%"
)

print(
    "Casos coincidentes:",
    auditoria_revision["acuerdo"].sum(),
    "de",
    len(auditoria_revision)
)

NameError: name 'auditoria_revision' is not defined

## 3. División de los datos

La base se divide de forma estratificada en:

- 70 % entrenamiento
- 15 % validación
- 15 % prueba

Se utiliza `random_state = 42` para garantizar reproducibilidad.
El conjunto de prueba se mantiene aislado durante el entrenamiento.

In [2]:
# Unir las etiquetas originales con las etiquetas generadas por Ollama
df_modelo = df[
    ["titular", "etiqueta"]
].merge(
    df_llm[
        ["titular", "etiqueta_llm", "confianza"]
    ],
    on="titular",
    how="inner"
)

print("Dimensión para modelado:", df_modelo.shape)

NameError: name 'df' is not defined

In [ ]:
train_df, temp_df = train_test_split(
    df_modelo,
    test_size=0.30,
    stratify=df_modelo["etiqueta"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["etiqueta"],
    random_state=42
)

print("Entrenamiento:", len(train_df))
print("Validación:", len(val_df))
print("Prueba:", len(test_df))

print(
    "Total:",
    len(train_df) + len(val_df) + len(test_df)
)

Entrenamiento: 252
Validación: 54
Prueba: 54
Total: 360


In [ ]:
print("TRAIN")
print(train_df["etiqueta"].value_counts())
print()

print("VALIDACIÓN")
print(val_df["etiqueta"].value_counts())
print()

print("TEST")
print(test_df["etiqueta"].value_counts())

TRAIN
etiqueta
politica              67
economia              66
ciencia_tecnologia    66
deportes              53
Name: count, dtype: int64

VALIDACIÓN
etiqueta
economia              15
ciencia_tecnologia    14
politica              14
deportes              11
Name: count, dtype: int64

TEST
etiqueta
ciencia_tecnologia    15
economia              14
politica              14
deportes              11
Name: count, dtype: int64


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
LABELS = [
    "politica",
    "economia",
    "deportes",
    "ciencia_tecnologia"
]

label2id = {
    etiqueta: i
    for i, etiqueta in enumerate(LABELS)
}

id2label = {
    i: etiqueta
    for etiqueta, i in label2id.items()
}

label2id

{'politica': 0, 'economia': 1, 'deportes': 2, 'ciencia_tecnologia': 3}

In [ ]:
train_hf = train_df[
    ["titular", "etiqueta_llm"]
].copy()

val_hf = val_df[
    ["titular", "etiqueta_llm"]
].copy()

train_hf["label"] = train_hf["etiqueta_llm"].map(label2id)
val_hf["label"] = val_hf["etiqueta_llm"].map(label2id)

print("Nulos train:", train_hf["label"].isna().sum())
print("Nulos val:", val_hf["label"].isna().sum())

Nulos train: 0
Nulos val: 0


In [ ]:
train_dataset = Dataset.from_pandas(
    train_hf[["titular", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_hf[["titular", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

NameError: name 'Dataset' is not defined

In [3]:
MODELO_BASE = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODELO_BASE
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=4,
    label2id=label2id,
    id2label=id2label
)

NameError: name 'AutoTokenizer' is not defined

In [ ]:
def tokenizar(batch):
    return tokenizer(
        batch["titular"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_tokenized = train_dataset.map(
    tokenizar,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenizar,
    batched=True
)

print(train_tokenized)
print(val_tokenized)

Map: 100%|██████████| 54/54 [00:00<00:00, 6130.70 examples/s]

Dataset({
    features: ['titular', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 252
})
Dataset({
    features: ['titular', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 54
})


In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predicciones = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predicciones
    )

    f1 = f1_score(
        labels,
        predicciones,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1_macro": f1
    }

In [ ]:
%pip install -U accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
training_args = TrainingArguments(
    output_dir="modelo/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=42
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

NameError: name 'model' is not defined

In [5]:
import pandas as pd

df = pd.read_csv("datos/noticias.csv")
df_llm = pd.read_csv("datos/etiquetas_llm.csv")

print(df.shape)
print(df_llm.shape)

(360, 7)
(360, 5)


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

DATOS = Path("datos")

df = pd.read_csv(DATOS / "noticias.csv")
df_llm = pd.read_csv(DATOS / "etiquetas_llm.csv")

df_modelo = df[
    ["titular", "etiqueta"]
].merge(
    df_llm[
        ["titular", "etiqueta_llm", "confianza"]
    ],
    on="titular",
    how="inner"
)

LABELS = [
    "politica",
    "economia",
    "deportes",
    "ciencia_tecnologia"
]

label2id = {
    etiqueta: i
    for i, etiqueta in enumerate(LABELS)
}

id2label = {
    i: etiqueta
    for etiqueta, i in label2id.items()
}

train_df, temp_df = train_test_split(
    df_modelo,
    test_size=0.30,
    stratify=df_modelo["etiqueta"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["etiqueta"],
    random_state=42
)

train_hf = train_df[
    ["titular", "etiqueta_llm"]
].copy()

val_hf = val_df[
    ["titular", "etiqueta_llm"]
].copy()

train_hf["label"] = train_hf["etiqueta_llm"].map(label2id)
val_hf["label"] = val_hf["etiqueta_llm"].map(label2id)

train_dataset = Dataset.from_pandas(
    train_hf[["titular", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_hf[["titular", "label"]],
    preserve_index=False
)

print("Todo reconstruido correctamente")
print("Train:", len(train_dataset))
print("Validación:", len(val_dataset))
print("Test:", len(test_df))

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Todo reconstruido correctamente
Train: 252
Validación: 54
Test: 54


In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODELO_BASE = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODELO_BASE
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=4,
    label2id=label2id,
    id2label=id2label
)

print("Modelo cargado correctamente")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1614.14it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado correctamente


In [8]:
def tokenizar(batch):
    return tokenizer(
        batch["titular"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenizar,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenizar,
    batched=True
)

print("Train tokenizado:", len(train_tokenized))
print("Validación tokenizada:", len(val_tokenized))

Map: 100%|██████████| 54/54 [00:00<00:00, 6769.86 examples/s]

Train tokenizado: 252
Validación tokenizada: 54


In [9]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predicciones)
    f1 = f1_score(labels, predicciones, average="macro")

    return {
        "accuracy": accuracy,
        "f1_macro": f1
    }

training_args = TrainingArguments(
    output_dir="modelo/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=42
)

print("Configuración lista")

Configuración lista


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer listo")

Trainer listo


In [13]:
df_llm_prueba = pd.read_csv("datos/etiquetas_llm.csv")

print("Dimensión:", df_llm_prueba.shape)

print("\nEtiquetas guardadas:")
print(
    df_llm_prueba["etiqueta_llm"]
    .value_counts(dropna=False)
)

df_llm_prueba.head()

Dimensión: (360, 5)

Etiquetas guardadas:
etiqueta_llm
NaN    360
Name: count, dtype: int64


,titular,etiqueta_llm,confianza,modelo_llm,fecha
0,Presidente del TC sobre leyes aprobadas por el...,NaN,NaN,qwen2.5:3b,2026-09-07T20:51:41-05:00
1,Caso Árbitros-Odebrecht: audiencia de control ...,NaN,NaN,qwen2.5:3b,2026-10-12T12:48:23-05:00
2,Flavio Figallo del Partido del Buen Gobierno: ...,NaN,NaN,qwen2.5:3b,2026-10-16T11:47:20-05:00
3,Unión Europea y Minem acuerdan creación de mes...,NaN,NaN,qwen2.5:3b,2026-10-13T19:19:04-05:00
4,Fiscalía realizó diligencias en Ministerio de ...,NaN,NaN,qwen2.5:3b,2026-08-27T13:43:00-05:00


In [14]:
import pandas as pd
import json
import unicodedata
import ollama
from pathlib import Path

DATOS = Path("datos")

# Volver a cargar la base original
df = pd.read_csv(DATOS / "noticias.csv")


def normalizar_etiqueta(texto):

    if texto is None:
        return None

    texto = str(texto).strip().lower()

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    if "politic" in texto:
        return "politica"

    if "econom" in texto or "negocio" in texto:
        return "economia"

    if "deport" in texto:
        return "deportes"

    if "tecnolog" in texto or "ciencia" in texto:
        return "ciencia_tecnologia"

    return None


def clasificar_ollama(titular):

    prompt = f"""
Clasifica este titular de noticia peruana.

Solo puedes utilizar UNA de estas etiquetas exactas:

politica
economia
deportes
ciencia_tecnologia

Devuelve solamente JSON:

{{
    "etiqueta": "politica",
    "confianza": 0.95
}}

Titular:
{titular}
"""

    respuesta = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        format="json",
        options={"temperature": 0}
    )

    resultado = json.loads(
        respuesta["message"]["content"]
    )

    etiqueta = normalizar_etiqueta(
        resultado.get("etiqueta")
    )

    confianza = float(
        resultado.get("confianza", 0)
    )

    return etiqueta, confianza


resultados = []

for i, fila in df.iterrows():

    etiqueta = None
    confianza = None

    # Hasta 3 intentos si responde algo extraño
    for intento in range(3):

        try:
            etiqueta, confianza = clasificar_ollama(
                fila["titular"]
            )

            if etiqueta is not None:
                break

        except Exception:
            pass

    resultados.append({
        "titular": fila["titular"],
        "etiqueta_llm": etiqueta,
        "confianza": confianza,
        "modelo_llm": "qwen2.5:3b",
        "fecha": fila["fecha"]
    })

    if (i + 1) % 20 == 0:
        print(f"Procesadas {i + 1} de {len(df)}")


df_llm = pd.DataFrame(resultados)

df_llm.to_csv(
    DATOS / "etiquetas_llm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTerminado.")
print("Dimensión:", df_llm.shape)
print("Etiquetas vacías:", df_llm["etiqueta_llm"].isna().sum())

print("\nDistribución:")
print(df_llm["etiqueta_llm"].value_counts(dropna=False))

Procesadas 20 de 360
Procesadas 40 de 360
Procesadas 60 de 360
Procesadas 80 de 360
Procesadas 100 de 360
Procesadas 120 de 360
Procesadas 140 de 360
Procesadas 160 de 360
Procesadas 180 de 360
Procesadas 200 de 360
Procesadas 220 de 360
Procesadas 240 de 360
Procesadas 260 de 360
Procesadas 280 de 360
Procesadas 300 de 360
Procesadas 320 de 360
Procesadas 340 de 360
Procesadas 360 de 360

Terminado.
Dimensión: (360, 5)
Etiquetas vacías: 5

Distribución:
etiqueta_llm
politica              138
deportes               86
ciencia_tecnologia     66
economia               65
NaN                     5
Name: count, dtype: int64


In [15]:
pendientes = df_llm[df_llm["etiqueta_llm"].isna()].copy()

pendientes[["titular", "fecha"]]

,titular,fecha
50,"Curiosos con causa, una crónica de Fernando Vi...",2026-09-14T09:45:02-05:00
274,Informe Planeta Vivo 2026: la fauna silvestre ...,2026-10-10T01:08:34-05:00
277,¿Qué secreto de longevidad atesora una tortuga...,2026-10-10T21:05:00-05:00
313,¡Por primera vez en el Clausura! Piero Cari se...,2026-10-16T20:11:13-05:00
344,"Físico, compromiso y entorno: las razones que ...",2026-10-17T12:25:28-05:00


In [16]:
correcciones = {
    50: "politica",
    274: "ciencia_tecnologia",
    277: "ciencia_tecnologia",
    313: "deportes",
    344: "deportes"
}

for idx, etiqueta in correcciones.items():
    df_llm.loc[idx, "etiqueta_llm"] = etiqueta

df_llm.to_csv(
    DATOS / "etiquetas_llm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Etiquetas vacías:", df_llm["etiqueta_llm"].isna().sum())

Etiquetas vacías: 0


In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

DATOS = Path("datos")

df = pd.read_csv(DATOS / "noticias.csv")
df_llm = pd.read_csv(DATOS / "etiquetas_llm.csv")

df_modelo = df[
    ["titular", "etiqueta"]
].merge(
    df_llm[
        ["titular", "etiqueta_llm", "confianza"]
    ],
    on="titular",
    how="inner"
)

LABELS = [
    "politica",
    "economia",
    "deportes",
    "ciencia_tecnologia"
]

label2id = {etiqueta: i for i, etiqueta in enumerate(LABELS)}
id2label = {i: etiqueta for etiqueta, i in label2id.items()}

train_df, temp_df = train_test_split(
    df_modelo,
    test_size=0.30,
    stratify=df_modelo["etiqueta"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["etiqueta"],
    random_state=42
)

train_hf = train_df[
    ["titular", "etiqueta_llm"]
].copy()

val_hf = val_df[
    ["titular", "etiqueta_llm"]
].copy()

train_hf["label"] = (
    train_hf["etiqueta_llm"]
    .map(label2id)
    .astype(int)
)

val_hf["label"] = (
    val_hf["etiqueta_llm"]
    .map(label2id)
    .astype(int)
)

print("Nulos train:", train_hf["label"].isna().sum())
print("Nulos val:", val_hf["label"].isna().sum())
print("Train:", len(train_hf))
print("Val:", len(val_hf))
print("Test:", len(test_df))

Nulos train: 0
Nulos val: 0
Train: 252
Val: 54
Test: 54


In [19]:
train_dataset = Dataset.from_pandas(
    train_hf[["titular", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_hf[["titular", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['titular', 'label'],
    num_rows: 252
})
Dataset({
    features: ['titular', 'label'],
    num_rows: 54
})


In [20]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODELO_BASE = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=4,
    label2id=label2id,
    id2label=id2label
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 905.25it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
def tokenizar(batch):
    return tokenizer(
        batch["titular"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenizar,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenizar,
    batched=True
)

print("Train tokenizado:", len(train_tokenized))
print("Val tokenizado:", len(val_tokenized))

Map: 100%|██████████| 54/54 [00:00<00:00, 3594.72 examples/s]

Train tokenizado: 252
Val tokenizado: 54


In [22]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predicciones),
        "f1_macro": f1_score(labels, predicciones, average="macro")
    }

training_args = TrainingArguments(
    output_dir="modelo/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer listo")

Trainer listo


In [23]:
trainer.train()

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.300616,1.209074,0.500000,0.343698
2,0.990769,0.917757,0.666667,0.621453
3,0.830658,0.824805,0.777778,0.773489


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]
c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]


TrainOutput(global_step=96, training_loss=1.0779393464326859, metrics={'train_runtime': 177.8395, 'train_samples_per_second': 4.251, 'train_steps_per_second': 0.54, 'total_flos': 25037231321088.0, 'train_loss': 1.0779393464326859, 'epoch': 3.0})

In [24]:
test_hf = test_df[["titular", "etiqueta"]].copy()

test_hf["label"] = (
    test_hf["etiqueta"]
    .map(label2id)
    .astype(int)
)

test_dataset = Dataset.from_pandas(
    test_hf[["titular", "label"]],
    preserve_index=False
)

test_tokenized = test_dataset.map(
    tokenizar,
    batched=True
)

resultado_test = trainer.predict(test_tokenized)

predicciones_test = np.argmax(
    resultado_test.predictions,
    axis=-1
)

accuracy_final = accuracy_score(
    test_hf["label"],
    predicciones_test
)

f1_final = f1_score(
    test_hf["label"],
    predicciones_test,
    average="macro"
)

print("Accuracy final:", round(accuracy_final, 4))
print("F1-macro final:", round(f1_final, 4))

Map: 100%|██████████| 54/54 [00:00<00:00, 560.95 examples/s]
c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy final: 0.6481
F1-macro final: 0.6656


In [25]:
from sklearn.metrics import confusion_matrix, classification_report
import plotly.express as px

cm = confusion_matrix(
    test_hf["label"],
    predicciones_test
)

fig = px.imshow(
    cm,
    text_auto=True,
    x=LABELS,
    y=LABELS,
    labels={
        "x": "Predicción",
        "y": "Etiqueta real",
        "color": "Cantidad"
    },
    title="Matriz de confusión del modelo final"
)

fig.show(renderer="browser")

print(
    classification_report(
        test_hf["label"],
        predicciones_test,
        target_names=LABELS
    )
)

                    precision    recall  f1-score   support

          politica       0.42      1.00      0.60        14
          economia       1.00      0.50      0.67        14
          deportes       1.00      0.82      0.90        11
ciencia_tecnologia       1.00      0.33      0.50        15

          accuracy                           0.65        54
         macro avg       0.86      0.66      0.67        54
      weighted avg       0.85      0.65      0.65        54



In [26]:
accuracy_llm = accuracy_score(
    df_modelo["etiqueta"],
    df_modelo["etiqueta_llm"]
)

f1_llm = f1_score(
    df_modelo["etiqueta"],
    df_modelo["etiqueta_llm"],
    average="macro"
)

print("Accuracy Ollama:", round(accuracy_llm, 4))
print("F1-macro Ollama:", round(f1_llm, 4))

Accuracy Ollama: 0.7333
F1-macro Ollama: 0.731


In [27]:
comparacion = pd.DataFrame({
    "Modelo": [
        "Zero-shot",
        "Ollama Qwen 2.5 3B",
        "DistilBERT fine-tuned"
    ],
    "Accuracy": [
        0.6500,
        accuracy_llm,
        accuracy_final
    ],
    "F1_macro": [
        0.6328,
        f1_llm,
        f1_final
    ]
})

comparacion

,Modelo,Accuracy,F1_macro
0,Zero-shot,0.650000,0.632800
1,Ollama Qwen 2.5 3B,0.733333,0.731017
2,DistilBERT fine-tuned,0.648148,0.665603


### Reflexión sobre los resultados

Los resultados muestran que el modelo Ollama Qwen 2.5 3B obtuvo el mejor desempeño, con una Accuracy de 73.33% y un F1 Macro de 73.10%. El modelo Zero-shot alcanzó una Accuracy de 65.00% y un F1 Macro de 63.28%, mientras que DistilBERT ajustado obtuvo una Accuracy de 64.81% y un F1 Macro de 66.56%.

Aunque el fine-tuning de DistilBERT permitió mejorar el F1 Macro respecto al enfoque Zero-shot, no logró superar al modelo Ollama. La matriz de confusión mostró que DistilBERT clasificó correctamente la mayoría de noticias de política y deportes, pero presentó dificultades especialmente en ciencia y tecnología, donde varias noticias fueron confundidas con política.

Una posible explicación es el tamaño reducido del conjunto de datos y el uso de etiquetas generadas mediante supervisión débil. Con una mayor cantidad de noticias y una revisión manual más amplia de las etiquetas, el desempeño del modelo fine-tuned podría mejorar.

In [28]:
trainer.save_model("modelo")
tokenizer.save_pretrained("modelo")

print("Modelo y tokenizer guardados correctamente.")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.16s/it]

Modelo y tokenizer guardados correctamente.


In [29]:
from pathlib import Path

for archivo in Path("modelo").iterdir():
    print(archivo.name)
    

checkpoints
config.json
model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin
